# Enhanced Summary Knowledge Tuning - Data Generation

## Overview

This notebook demonstrates how to generate high-quality knowledge tuning datasets using the SDG Hub framework. It creates multiple types of document augmentations and corresponding question-answer pairs that can be used to train or fine-tune language models for enhanced summarization and knowledge extraction capabilities.

## What This Notebook Does

This notebook will:

2. **Generate Four Types of Knowledge Tuning Datasets**:
   - **Extractive Summaries**: Concise summaries that extract key information directly from source documents
   - **Detailed Summaries**: Comprehensive summaries that provide thorough coverage of document content
   - **Key Facts**: Structured fact extraction with corresponding Q&A pairs
   - **Document-Based Q&A**: Question-answer pairs generated directly from document content


4. **Output Structured Training Data**:
   - For each augmentation we save JSONL dataset.
   - You can follow [knowledge_mixing](knowledge_mixing.ipynb) to convert it into training dataset

## Prerequisites

- SDG Hub installed and configured
- Environment variables set up (see [.env.example](.env.example)). Specifically set the model provider, seed data and output path.
- Document pre-processing completed (run [document_pre_processing.ipynb](document_pre_processing.ipynb) first)

```bash 
git clone https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub.git
cd sdg_hub
pip install .[examples]
copy the .env.example to .env and set the model endpoint and generation/mixing parameters
```
**⚠️ If you haven't already, run the document pre-processing notebook to create the seed data.**

## Next Steps

After running this notebook, use [knowledge_mixing](knowledge_mixing.ipynb) to combine and curate the generated datasets for final model training.


In [1]:
# Third Party
from datasets import load_dataset
from dotenv import load_dotenv

# First Party
from sdg_hub import Flow, FlowRegistry
from sdg_hub.core.utils.translation import translate_flow
import os
from pathlib import Path

# Load environment variables from .env file
load_dotenv()

/opt/app-root/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Required to run the flow with async mode
import nest_asyncio

nest_asyncio.apply()

In [3]:
def create_seed_data_from_quality_benchmark(
    run_on_validation=None, seed_data_path=None
):
    """
    Create seed data from QuALITY Benchmark dataset.

    Args:
        run_on_validation (bool, optional): If True, use validation subset. If None, reads from env.
        seed_data_path (str, optional): Path to save seed data. If None, reads from env.

    Returns:
        datasets.Dataset: The processed corpus
    """
    # Use environment variables as defaults if not provided
    if run_on_validation is None:
        run_on_validation = os.getenv("RUN_ON_VALIDATION_SET", "true").lower() == "true"
    if seed_data_path is None:
        seed_data_path = os.getenv("SEED_DATA_PATH", "seed_data_val.jsonl")

    # Load QuALITY Benchmark dataset
    print("Loading QuALITY Benchmark dataset...")
    quality_corpus = (
        load_dataset("zitongyang/entigraph-quality-corpus", split="train")
        .remove_columns(["entity", "entigraph"])
        .rename_columns({"raw": "document", "uid": "document_outline"})
    )

    # Define seed examples for knowledge tuning
    seed_examples = {
        "icl_document": (
            "The coastal town of Willow Creek, once renowned for its pristine beaches, now struggles with rampant pollution. Plastic debris and oil spills have devastated marine life, prompting a decline in tourism and fishing industries. Residents have organized weekly clean-up initiatives, but the scale of the problem overwhelms their efforts.",
            "Technologists at the local university have developed an AI-powered buoy system to combat this. The buoys, equipped with solar panels and filtration technology, can identify and absorb oil spills while collecting microplastics. Data from the buoys is shared publicly, raising awareness and pressuring corporations to adopt sustainable practices. Though costly, the project has sparked hope for revitalizing the ecosystem and economy.",
        ),
        "icl_query_1": "How does the technological solution address the economic *and* environmental challenges highlighted in the document?",
        "icl_query_2": "What implicit values or priorities do the community's actions (clean-up initiatives) and the technologists' project reflect, and how do these align or contrast?",
        "icl_query_3": "Imagine the buoy project succeeds. What unintended consequences might arise from its impact, considering document's themes?",
        "domain": "articles/essays",
    }

    # Add seed examples to the corpus
    quality_corpus = quality_corpus.map(lambda x: seed_examples)

    if run_on_validation:
        # Validation set - use predefined document IDs for consistent evaluation
        DOC_UIDS = [
            " Defining Decay Down by David Plotz",
            " Fight Clubbed by David Plotz",
            " I, Antichrist? by Jeffrey Goldberg",
            " It's Time To Keelhaul U-Haul! by Jeffrey Goldberg",
            " My Father's Estate by Ben Stein",
            '"Phone Me in Central Park" by McConnell, James V.',
            "A Coffin for Jacob by Ludwig, Edward W.",
            "A Fall of Glass by Lee, Stanley R.",
            "A Filbert Is a Nut by Raphael, Rick",
            "A Gift from Earth by Banister, Manly",
            "A Gleeb for Earth by Schafhauser, Charles",
            "A Good Year for the Roses? by David Edelstein",
            "A Pail of Air by Leiber, Fritz",
            "A Planet Named Joe by Hunter, Evan",
            "AI: what's the worst that could happen? by Harry Armstrong",
            "Accidental Death by Baily, Peter",
            "All Day September by Kuykendall, Roger",
            "Ambition by Bade, William L.",
            "And Then the Town Took Off by Wilson, Richard",
            "Atom Mystery [Young Atom Detective] by Coombs, Charles Ira",
            "Beach Scene by King, Marshall",
            "Big Ancestor by Wallace, F. L. (Floyd L.)",
            "Birds of a Feather by Silverberg, Robert",
            "Bodyguard by Gold, H. L. (Horace Leonard)",
        ]

        # Filter corpus to validation set
        quality_corpus = quality_corpus.filter(
            lambda x: x["document_outline"] in DOC_UIDS
        )
        print(f"Running on validation set with {len(quality_corpus)} documents")
    else:
        # Use full dataset for training
        print(f"Running on full dataset with {len(quality_corpus)} documents")

    # Save the seed data
    quality_corpus.to_json(seed_data_path, orient="records", lines=True)
    print(f"Saved seed data to: {seed_data_path}")

    return quality_corpus

In [4]:
# Load seed data. If one is not provided, create it from the quality benchmark dataset.
seed_data_path = os.getenv("SEED_DATA_PATH", "seed_data.jsonl")

if not os.path.exists(seed_data_path):
    print(f"{seed_data_path} not found. Creating seed data...")
    quality_corpus = create_seed_data_from_quality_benchmark(
        seed_data_path=seed_data_path
    )
else:
    print(f"Loading existing seed data from {seed_data_path}")
    quality_corpus = load_dataset("json", data_files=seed_data_path, split="train")

# Subsample the seed data. Useful for debugging.
subsample = int(os.getenv("SEED_DATA_SUBSAMPLE", "0"))
if subsample > 0:
    dataset_size = len(quality_corpus)
    effective_subsample = min(subsample, dataset_size)
    if effective_subsample < subsample:
        print(
            f"SEED_DATA_SUBSAMPLE={subsample} is greater than dataset size={dataset_size}. "
            f"Using {effective_subsample} instead."
        )
    quality_corpus = quality_corpus.select(range(effective_subsample))

quality_corpus = quality_corpus.to_pandas()


Loading existing seed data from seed_data.jsonl
SEED_DATA_SUBSAMPLE=24 is greater than dataset size=1. Using 1 instead.


### Run SDG
- This will create knowledge flow from provided yaml file
- We will run this on small dataset for demo purposes
- For large scale generation, please use the python command provided in the next cell
- You can analyze the generated data to ensure the quality is similar to proivded QnA pairs

In [5]:
# Setup model configuration in flow object
def set_model_config(flow_object):
    model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")
    print(f"Using model provider: {model_provider}")
    # Set model provider
    if model_provider == "hosted_vllm":
        vllm_model = os.getenv(
            "VLLM_MODEL", "hosted_vllm/meta-llama/Llama-3.3-70B-Instruct"
        )
        vllm_api_base = os.getenv("VLLM_API_BASE", "http://localhost:8000/v1")
        vllm_api_key = os.getenv("VLLM_API_KEY", "EMPTY")
        enable_reasoning = os.getenv("ENABLE_REASONING", "false").lower() in (
            "1",
            "true",
            "yes",
        )
        print(f"Using reasoning: {enable_reasoning}")
        flow_object.set_model_config(
            model=vllm_model,
            api_base=vllm_api_base,
            api_key=vllm_api_key,
            enable_reasoning=enable_reasoning,
        )
    elif model_provider == "openai":
        openai_api_key = os.getenv("OPENAI_API_KEY")
        openai_model = os.getenv("OPENAI_MODEL", "openai/gpt-4")
        flow_object.set_model_config(
            model=openai_model,
            api_key=openai_api_key,
        )
    elif model_provider == "openrouter":
        openai_api_key = os.getenv("OPENAI_API_KEY")
        openai_model = os.getenv("OPENAI_MODEL", "openai/gpt-4")
        flow_object.set_model_config(
            model=openai_model,
            api_key=openai_api_key,
            api_base="https://openrouter.ai/api/v1",
        )
    elif model_provider == "ollama":
        ollama_model = os.getenv("OLLAMA_MODEL", "ollama/gemma2")
        ollama_api_base = os.getenv("OLLAMA_API_BASE", "http://localhost:11434")
        flow_object.set_model_config(
            model=ollama_model,
            api_base=ollama_api_base,
        )
    elif model_provider == "maas":
        maas_model = os.getenv("MAAS_MODEL")
        maas_api_base = os.getenv("MAAS_API_BASE")
        maas_api_key = os.getenv("MAAS_API_KEY")
        flow_object.set_model_config(
            model=maas_model,
            api_base=maas_api_base,
            api_key=maas_api_key,
        )
    return flow_object

#### Discover the available generation flows

In [6]:
# Auto-discover all available flows (no setup needed!)
FlowRegistry.discover_flows()

# List available flows
flows = FlowRegistry.list_flows()
print(f"Available flows: {flows}")

# You can also search the flows by tag
qa_flows = FlowRegistry.search_flows(tag="question-generation")
print(f"QA flows: {qa_flows}")

[20:43:00] INFO     Discovered 11 flows                                                             ]8;id=527960;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/registry.py\registry.py]8;;\:]8;id=264222;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/registry.py#126\126]8;;\

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID                  ┃ Name                 ┃ Author               ┃ Tags                 ┃ Description          ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ clean-shadow-397    │ Advanced Japanese    │ SDG Hub Contributors │ question-generation, │ A comprehensive flow │
│                     │ Document Grounded    │                      │ knowledge-extractio… │ that generates       │
│                     │ Question-Answer      │                      │ qa-pairs,            │ high-quality         │
│                     │ Generation Flow for  │                      │ document-processing, │ question-answer      │
│                     │ Knowledge Tuning     │                      │ educational,         │ pairs from Japanese  │
│                     │                      │                      │ multilingual,        │ input documents      │
│                     │                      │                      │ japanese             │ using multiple LLM   │
│                     │                      │                      │                      │ blocks for question  │
│                     │                      │                      │                      │ generation, answer   │
│                     │                      │                      │                      │ synthesis, and       │
│                     │                      │                      │                      │ quality evaluation.  │
│ epic-jade-656       │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document. Each │
│                     │ Flow                 │                      │ knowledge-extractiv… │ document is first    │
│                     │                      │                      │ qa-pairs,            │ converted into list  │
│                     │                      │                      │ extractive-summaries │ of knowledge         │
│                     │                      │                      │                      │ segments for         │
│                     │                      │                      │                      │ creating extractive  │
│                     │                      │                      │                      │ summary and then     │
│                     │                      │                      │                      │ annotated with       │
│                     │                      │                      │                      │ context,             │
│                     │                      │                      │                      │ relationship and     │
│                     │                      │                      │                      │ relevance. This is   │
│                     │                      │                      │                      │ then converted into  │
│                     │                      │                      │                      │ Question-Answer      │
│                     │                      │                      │                      │ pairs.               │
│ epic-jade-656-es    │ Extractive Summary   │ SDG Hub Contributors │ knowledge-tuning,    │ Generate extractive  │
│                     │ Knowledge Tuning     │                      │ document-internaliz… │ summary from the     │
│                     │ Dataset Generation   │                      │ question-generation, │ input document in    │
│                     │ Flow (Spanish)       │                      │ knowledge-extractiv… │ Spanish. Each        │
│                     │                      │          

Available flows: [{'id': 'loud-dawn-245', 'name': 'RAG Evaluation Dataset Flow'}, {'id': 'clean-shadow-397', 'name': 'Advanced Japanese Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'mild-thunder-748', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'stellar-peak-605', 'name': 'Document Based Knowledge Tuning Dataset Generation Flow'}, {'id': 'epic-jade-656', 'name': 'Extractive Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'heavy-heart-77', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow'}, {'id': 'mild-thunder-748-es', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'stellar-peak-605-es', 'name': 'Document Based Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'epic-jade-656-es', 'name': 'Extractive Summary Knowledge Tuning Dataset Generation Flow (Spanish)'}, {'id': 'heavy-heart-77-es', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow

#### Multilingual Support (Optional)

If `SDG_LANG` and `SDG_LANG_CODE` are set in your `.env` file, the notebook will
use translated flow variants. If translated flows are not yet available in the
FlowRegistry, they will be created automatically using `translate_flow()`.

Delete `SDG_LANG` from your `.env` to use the default English flows.

In [7]:
# Read multilingual settings from .env
sdg_lang = os.getenv("SDG_LANG", "").strip()
sdg_lang_code = os.getenv("SDG_LANG_CODE", "").strip()

if sdg_lang and not sdg_lang_code:
    raise ValueError("SDG_LANG is set but SDG_LANG_CODE is missing. Both are required.")
if sdg_lang_code and not sdg_lang:
    raise ValueError("SDG_LANG_CODE is set but SDG_LANG is missing. Both are required.")

# If a custom translated flows directory exists, register it for discovery
translated_flows_dir = os.getenv("TRANSLATED_FLOWS_DIR", "").strip()
if translated_flows_dir and os.path.isdir(translated_flows_dir):
    FlowRegistry.register_search_path(translated_flows_dir)
    FlowRegistry._discover_flows(force_refresh=True)


def ensure_translated_flow(base_flow_name: str) -> str:
    """Return the flow name to use, translating on-demand if needed.

    When SDG_LANG is unset the base (English) name is returned unchanged.
    Otherwise checks FlowRegistry for a pre-existing translated variant
    and falls back to running translate_flow() to create one.
    """
    if not sdg_lang:
        return base_flow_name

    translated_name = f"{base_flow_name} ({sdg_lang})"
    if FlowRegistry.get_flow_path(translated_name) is not None:
        print(f"  ✓ Found: {translated_name}")
        return translated_name

    # Translate on-demand
    print(f"  ⟳ Translating: {base_flow_name} → {sdg_lang}...")
    # Compute per-flow output subdir to avoid flow.yaml collisions
    source_path = FlowRegistry.get_flow_path(base_flow_name)
    flow_subdir = Path(source_path).parent.name
    output_dir = None
    if translated_flows_dir:
        output_dir = str(Path(translated_flows_dir) / f"{flow_subdir}_{sdg_lang_code}")

    translate_flow(
        flow=base_flow_name,
        lang=sdg_lang,
        lang_code=sdg_lang_code,
        translator_model=os.getenv("TRANSLATOR_MODEL", "openai/gpt-4o"),
        verifier_model=os.getenv(
            "VERIFIER_MODEL", os.getenv("TRANSLATOR_MODEL", "openai/gpt-4o")
        ),
        output_dir=output_dir,
        translator_api_key=os.getenv("TRANSLATOR_API_KEY"),
        translator_api_base=os.getenv("TRANSLATOR_API_BASE"),
        verifier_api_key=os.getenv("VERIFIER_API_KEY"),
        verifier_api_base=os.getenv("VERIFIER_API_BASE"),
        register=True,
    )
    return translated_name


# Resolve all flow names (translating if needed)
BASE_FLOW_NAMES = [
    "Extractive Summary Knowledge Tuning Dataset Generation Flow",
    "Detailed Summary Knowledge Tuning Dataset Generation Flow",
    "Key Facts Knowledge Tuning Dataset Generation Flow",
    "Document Based Knowledge Tuning Dataset Generation Flow",
]

if sdg_lang:
    print(f"Multilingual mode: {sdg_lang} ({sdg_lang_code})\n")
else:
    print("Language: English (default)\n")

flow_names = {}
for name in BASE_FLOW_NAMES:
    flow_names[name] = ensure_translated_flow(name)

print(f"\nFlows to use: {list(flow_names.values())}")

Language: English (default)


Flows to use: ['Extractive Summary Knowledge Tuning Dataset Generation Flow', 'Detailed Summary Knowledge Tuning Dataset Generation Flow', 'Key Facts Knowledge Tuning Dataset Generation Flow', 'Document Based Knowledge Tuning Dataset Generation Flow']


In [8]:
# Get runtime parameters
enable_reasoning = os.getenv("ENABLE_REASONING", "false").lower() in (
    "1",
    "true",
    "yes",
)
number_of_summaries = int(os.getenv("NUMBER_OF_SUMMARIES", "50"))
max_concurrency = int(os.getenv("MAX_CONCURRENCY", "50"))
save_data_path = os.getenv("OUTPUT_DATA_FOLDER", "")
generate_cpt = os.getenv("GENERATE_CPT", "false").lower() in (
    "1",
    "true",
    "yes",
)

In [9]:
# Generate data for extractive summary
flow_name = flow_names["Extractive Summary Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)
number_of_summaries = int(os.getenv("NUMBER_OF_SUMMARIES", "50"))
# Remove all blocks after document augmentation
if generate_cpt:
    kept_blocks = []
    for block in flow.blocks:
        if block.block_name == "question_generation_prompt":
            break
        kept_blocks.append(block)
    flow.blocks = kept_blocks
# Generate data for extractive summary
if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {
        "question_generation": {"max_tokens": 1024},
        "gen_extractive_summary": {"n": number_of_summaries, "max_tokens": 6000},
    }
else:
    runtime_params = {"gen_extractive_summary": {"n": number_of_summaries}}

extractive_summary_generated_data = flow.generate(
    quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency
)

os.makedirs(os.path.join(save_data_path, "extractive_summary"), exist_ok=True)

extractive_summary_generated_data.to_json(
    os.path.join(save_data_path, "extractive_summary" if not generate_cpt else "extractive_summary_cpt", "gen.jsonl"),
    orient="records",
    lines=True,
)

print(f"✓ Extractive summary: {len(extractive_summary_generated_data)} records")

print(f"✓ Columns: {list(extractive_summary_generated_data.columns.tolist())}")

[20:43:00] INFO     Loading flow from:                                                          ]8;id=819863;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/serialization.py\serialization.py]8;;\:]8;id=116875;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/serialization.py#60\60]8;;\
                    /opt/app-root/lib64/python3.12/site-packages/sdg_hub/flows/knowledge_infusi                    
                    on/enhanced_multi_summary_qa/extractive_summary/flow.yaml                                      

Using model provider: hosted_vllm
Using reasoning: False


[20:43:00] INFO     Auto-detected 4 LLM blocks for configuration: ['answer_generation',         ]8;id=900618;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=149318;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py#242\242]8;;\
                    'eval_faithful_llm_chat', 'gen_extractive_summary', 'question_generation']                     

           INFO     Successfully configured 4 LLM blocks with: model:                           ]8;id=72474;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=114882;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py#300\300]8;;\
                    'hosted_vllm/RedHatAI/gpt-oss-20b', api_base:                                                  
                    'http://granite-ai-inference-server-ocp.apps.cluster-jzfpx.jzfpx.sandbox251                    
                    8.opentlc.com/v1', api_key: (redacted), enable_reasoning: 'False'                              

           INFO     Configured blocks: ['answer_generation', 'eval_faithful_llm_chat',          ]8;id=3710;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py\model_config.py]8;;\:]8;id=561931;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/model_config.py#303\303]8;;\
                    'gen_extractive_summary', 'question_generation']                                               

...


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



[20:46:30] ERROR    Failed to generate async responses: litellm.Timeout: Timeout Error:       ]8;id=634663;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/blocks/llm/llm_chat_block.py\llm_chat_block.py]8;;\:]8;id=341310;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/blocks/llm/llm_chat_block.py#518\518]8;;\
                    Hosted_vllmException - <html><body><h1>504 Gateway Time-out</h1>                               
                    The server didn't respond in time.                                                             
                    </body></html>                                                                                 
                     LiteLLM Retried: 6 times                                                                      

[20:46:30] ERROR    Block 'gen_extractive_summary' failed during execution: litellm.Timeout:       ]8;id=224015;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py\execution.py]8;;\:]8;id=489881;file:///opt/app-root/lib64/python3.12/site-packages/sdg_hub/core/flow/execution.py#292\292]8;;\
                    Timeout Error: Hosted_vllmException - <html><body><h1>504 Gateway                              
                    Time-out</h1>                                                                                  
                    The server didn't respond in time.                                                             
                    </body></html>                                                                                 
                     LiteLLM Retried: 6 times                                                                      

╭───────────────────── Extractive Summary Knowledge Tuning Dataset Generation Flow - Failed ──────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ duplicate_document_… │ DuplicateColum… │      0.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ extractive_summary_… │ PromptBuilderB… │      0.01s │    1 → 1     │       +1        │     ✓      │           │
│ │ gen_extractive_summ… │ LLMChatBlock    │    210.26s │    1 → ❌    │        —        │     ✗      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 3 blocks        │    210.27s │   0 final    │     0 final     │    2/3     │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

FlowValidationError: Block 'gen_extractive_summary' execution failed: litellm.Timeout: Timeout Error: Hosted_vllmException - <html><body><h1>504 Gateway Time-out</h1>
The server didn't respond in time.
</body></html>
 LiteLLM Retried: 6 times

In [ ]:
# Generate similar data for Detailed Summary
flow_name = flow_names["Detailed Summary Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)

# Remove all blocks after document augmentation
if generate_cpt:
    kept_blocks = []
    for block in flow.blocks:
        if block.block_name == "question_generation_prompt":
            break
        kept_blocks.append(block)
    flow.blocks = kept_blocks

if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {
        "question_generation": {"max_tokens": 1024},
        "gen_detailed_summary": {"n": number_of_summaries, "max_tokens": 6000},
    }
else:
    runtime_params = {"gen_detailed_summary": {"n": number_of_summaries}}
# Generate data for detailed summary
detailed_summary_generated_data = flow.generate(
    quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency
)

os.makedirs(os.path.join(save_data_path, "detailed_summary"), exist_ok=True)

detailed_summary_generated_data.to_json(
    os.path.join(save_data_path, "detailed_summary" if not generate_cpt else "detailed_summary_cpt", "gen.jsonl"),
    orient="records",
    lines=True,
)

print(f"✓ Detailed summary: {len(detailed_summary_generated_data)} records")

print(f"✓ Columns: {list(detailed_summary_generated_data.columns.tolist())}")

In [ ]:
# Generate similar data for key facts
flow_name = flow_names["Key Facts Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)


# Remove all blocks after document augmentation
if generate_cpt:
    kept_blocks = []
    for block in flow.blocks:
        if block.block_name == "parse_atomic_facts_to_individual_facts":
            break
        kept_blocks.append(block)
    flow.blocks = kept_blocks

runtime_params = {}
if enable_reasoning:
    # Increase max tokens for Question Generation to accommodate reasoning content
    runtime_params = {
        "generate_key_fact_qa": {"max_tokens": 6000},
    }

# Generate data for key facts summary
key_facts_generated_data = flow.generate(
    quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency
)

os.makedirs(os.path.join(save_data_path, "key_facts_to_qa"), exist_ok=True)

key_facts_generated_data.to_json(
    os.path.join(save_data_path, "key_facts_to_qa" if not generate_cpt else "key_facts_cpt", "gen.jsonl"),
    orient="records",
    lines=True,
)

print(f"✓ Key facts: {len(key_facts_generated_data)} records")

print(f"✓ Columns: {list(key_facts_generated_data.columns.tolist())}")

In [ ]:
flow_name = flow_names["Document Based Knowledge Tuning Dataset Generation Flow"]
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)
runtime_params = {}
if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params = {
        "question_generation": {"max_tokens": 2048},
    }

if generate_cpt:
    # Skip generation if CPT is enabled. We simply train on the original document
    document_based_generated_data = quality_corpus[["document", "document_outline", "domain"]]
else:
    document_based_generated_data = flow.generate(
        quality_corpus, runtime_params=runtime_params, max_concurrency=max_concurrency
    )

os.makedirs(os.path.join(save_data_path, "document_based_qa"), exist_ok=True)

document_based_generated_data.to_json(
    os.path.join(save_data_path, "document_based_qa" if not generate_cpt else "document_based_cpt", "gen.jsonl"),
    orient="records",
    lines=True,
)

print(f"✓ Document based: {len(document_based_generated_data)} records")

print(f"✓ Columns: {list(document_based_generated_data.columns.tolist())}")

🎉 You now have all three four of document augmentations (detailed summaries, extractive summaries, key facts and document based) along with their corresponding QA pairs.

✅ Next steps:
   - Combine and curate these datasets to prepare your final training data.
   - For detailed guidance on post-processing, mixing, and formatting the data for model training (including conversion to messages format), please refer to [knowledge_mixing.ipynb](knowledge_mixing.ipynb).